In [0]:
alphacollector_transactions_history_path=dbutils.widgets.get("alphacollector_transactions_history_path")
mart_fact_adjustments_path=dbutils.widgets.get("mart_fact_adjustments_path")
alphacollector_claims_history_path=dbutils.widgets.get("alphacollector_claims_history_path")
office_path = dbutils.widgets.get("office_path")
client_path=dbutils.widgets.get("client_path")
payerdimension_path=dbutils.widgets.get("payerdimension_path")
adjustmentcode_path=dbutils.widgets.get("adjustmentcode_path")

In [0]:
spark.sql(
    f"""
CREATE OR REPLACE TEMPORARY VIEW adj AS
SELECT 
  EntryDate,
  TransCode,
  TransType,
  OfficeExternalId,
  ClaimNumber,
  SUM(TransAmt) AS AdjustmentAmt,
  ReportingWeekEndingDate
FROM {alphacollector_transactions_history_path}
WHERE TransType = 'Adjustment'
  AND ReportingWeekEndingDate = date_add(current_date(), -4)
GROUP BY 
  TransCode, TransType, OfficeExternalId, ClaimNumber, 
  ReportingWeekEndingDate, EntryDate;

  """
)



In [0]:
spark.sql(
    f"""
INSERT INTO {mart_fact_adjustments_path} (
    reporting_week_ending_date_key,
    posted_date_key,
    source_system_key,
    office_key,
    client_key,
    payor_key,
    adjustment_code_key,
    adjustment_type_key,
    invoice_number,
    adjustment_amount
)
SELECT 
  CAST(REPLACE(CAST(adj.ReportingWeekEndingDate AS STRING), '-', '') AS BIGINT) AS reporting_week_ending_date_key,
  CAST(CONCAT(
    SUBSTRING(adj.EntryDate, 7, 4),  -- Year from position 7-10
    '0',                              -- Literal '0'
    SUBSTRING(adj.EntryDate, 1, INSTR(adj.EntryDate, '/') - 1),  -- Month (before first /)
    SUBSTRING(adj.EntryDate, 4, 2)   -- Day from position 4-5
  ) AS BIGINT) AS posted_date_key,
  19 AS source_system_key,
  ofc.OfficeKey AS office_key,
  d.ClientKey AS client_key,
  f.PayerKey AS payor_key,
  g.Adjustment_Code_Key AS adjustment_code_key,
  NULL AS adjustment_type_key,
  adj.ClaimNumber AS invoice_number,
  CAST(adj.AdjustmentAmt AS DECIMAL(18,2)) AS adjustment_amount
  
FROM adj

LEFT JOIN {office_path} ofc 
  ON ofc.OfficeNumber = adj.OfficeExternalId
LEFT JOIN (
  SELECT ClaimNumber, MedicalRecordNumber, OfficeExternalId, PayerName
  FROM (
    SELECT 
      ClaimNumber, 
      MedicalRecordNumber, 
      OfficeExternalId, 
      PayerName,
      ROW_NUMBER() OVER (
        PARTITION BY ClaimNumber, OfficeExternalId 
        ORDER BY LoadDate DESC
      ) AS rnb
    FROM {alphacollector_claims_history_path}
  ) a
  WHERE rnb = 1
) b ON b.ClaimNumber = adj.ClaimNumber
   AND b.OfficeExternalId = adj.OfficeExternalId 

LEFT JOIN (
  SELECT ClientKey, MedicalRecordNumber
  FROM (
    SELECT 
      ClientKey, 
      MedicalRecordNumber,
      ROW_NUMBER() OVER (
        PARTITION BY MedicalRecordNumber 
        ORDER BY ClientKey DESC
      ) AS rnb
    FROM {client_path}
    WHERE SourceSystem = 'CUBHUB'
  ) c
  WHERE rnb = 1
) d ON b.MedicalRecordNumber = d.MedicalRecordNumber

LEFT JOIN (
  SELECT PayerKey, Name
  FROM (
    SELECT 
      PayerKey,
      Name,
      ROW_NUMBER() OVER (
        PARTITION BY Name 
        ORDER BY PayerKey DESC
      ) AS rnb
    FROM {payerdimension_path}
    WHERE SourceSystemKey = 19
  ) e
  WHERE rnb = 1
) f ON f.Name = b.PayerName

LEFT JOIN {adjustmentcode_path} g 
  ON g.Adjustment_Code_Description = adj.TransCode
  AND g.Source_System = 'CUBHUB';

    """
)